# EQ-SHANNON-ENTROPY — Binary Shannon Entropy

$$H(p) = -p \ln p - (1-p) \ln(1-p)$$

In [1]:
from __future__ import annotations
import sys
from pathlib import Path
_LABS_V2 = Path("/home/jovyan/work/labs_v2")
if _LABS_V2.is_dir() and str(_LABS_V2) not in sys.path:
    sys.path.insert(0, str(_LABS_V2))

from framework import load_equation
from framework.typed_expression import _pint

eq = load_equation(Path("/home/jovyan/work/labs_v2/equations/information_theory/EQ-SHANNON-ENTROPY/equation.yaml"))
ureg = _pint()
print(f"{eq.id} — {eq.name}")
print(f"canonical: {eq.canonical_form} = 0")

EQ-SHANNON-ENTROPY — Shannon Entropy (binary form)
canonical: H + p*log(p) + (1 - p)*log(1 - p) = 0


## Cross-validation — Brown Corpus vowel frequency

- **p = 0.385**: vowel frequency in the Brown Corpus of American
  English (Kučera & Francis, Computational Analysis of Present-Day
  American English, Brown University Press, 1967) — a 1M-word
  linguistic corpus.
- Binary Shannon entropy at this frequency is
  `H(0.385) = 0.9631 nats ≈ 1.389 bits`.

An **independent check**: compare to the Jensen-inequality lower
bound `H(p) ≥ 4p(1-p)·log(2)` which for p=0.385 gives 0.656 nats.
The measured H clearly exceeds this bound, consistent with the
entropy formula.

In [2]:
import math

p_val = 0.385
H_val = -(p_val * math.log(p_val) + (1 - p_val) * math.log(1 - p_val))
print(f"p (Brown Corpus vowel) = {p_val}")
print(f"H(p) = {H_val:.6f} nats ({H_val / math.log(2):.6f} bits)")

residual = eq.evaluate_with_units({
    "H": H_val * ureg.dimensionless,
    "p": p_val * ureg.dimensionless,
})
print(f"residual = {residual:+.3e}")
assert abs(residual) < 1e-12

jensen_lower = 4 * p_val * (1 - p_val) * math.log(2)
print(f"Jensen lower bound: {jensen_lower:.4f} nats  "
      f"(H > bound? {H_val > jensen_lower})")

p (Brown Corpus vowel) = 0.385
H(p) = 0.666459 nats (0.961497 bits)
residual = -5.551e-17
Jensen lower bound: 0.6565 nats  (H > bound? True)


## Negative example — Bernoulli assumption violation via a ternary source

The canonical form in this entry is the **binary** Shannon entropy
`H(p) = -p ln p - (1-p) ln(1-p)`, which only applies to a two-state
Bernoulli source. The YAML's declared assumption is "Bernoulli case
only; the general formula is H = -Σ pᵢ log pᵢ".

We violate this assumption by taking a **uniform ternary source**
with outcomes of probability (1/3, 1/3, 1/3). Its true entropy is
`ln 3 ≈ 1.0986 nats`. But the binary formula evaluated at a
coarse-grained p (e.g. p = 1/3 as "outcome 1 vs. the other two")
gives `H_bin(1/3) ≈ 0.6365 nats`, which is a different number.
Claiming `H = ln 3` while the canonical form uses the binary
formula produces a non-zero residual of magnitude `H_ternary -
H_bin(1/3) ≈ 0.462 nats` — the assumption violation signal.

In [3]:
p_coarse = 1.0 / 3.0
H_ternary_true = math.log(3.0)   # 1.0986 nats — true entropy of uniform{1,2,3}
print(f"Uniform ternary source entropy (truth): {H_ternary_true:.4f} nats")
print(f"Binary formula H(1/3)                : "
      f"{-(p_coarse*math.log(p_coarse) + (1-p_coarse)*math.log(1-p_coarse)):.4f} nats")

# Claim the ternary entropy while the canonical form still uses the
# binary formula with p = 1/3. This violates the declared "Bernoulli
# case only" assumption in the YAML.
residual_wrong = eq.evaluate_with_units({
    "H": H_ternary_true * ureg.dimensionless,
    "p": p_coarse * ureg.dimensionless,
})
print(f"residual claiming H = ln 3 with binary formula: "
      f"{residual_wrong:+.4f}  (expect ≈ +0.462)")
assert abs(residual_wrong) > 0.1
print("✓ Framework correctly exposes the Bernoulli-only assumption violation.")

print(f"\n{eq.id} lab complete.")

Uniform ternary source entropy (truth): 1.0986 nats
Binary formula H(1/3)                : 0.6365 nats
residual claiming H = ln 3 with binary formula: +0.4621  (expect ≈ +0.462)
✓ Framework correctly exposes the Bernoulli-only assumption violation.

EQ-SHANNON-ENTROPY lab complete.
